# Toxicity Guardrail Step Demo

This notebook walks through a simple example of using the toxicity-guardrail step in serving functions,
by first downloading the step from the hub and inspecting it, then including the step in a serving graph.

## Get the step from the hub

In [ ]:
import mlrun

In [ ]:
hub_step = mlrun.get_hub_step("hub://toxicity_guardrail")
hub_step.to_dict()

{'filename': 'toxicity_guardrail.py',
 'example': 'toxicity_guardrail.ipynb',
 'local_path': PosixPath('/User'),
 'url': 'hub://toxicity_guardrail',
 'class_name': 'ToxicityGuardrailStep',
 'name': 'toxicity_guardrail',
 'version': '1.0.0',
 'categories': ['data-preparation', 'model-serving', 'genai'],
 'description': 'Filters toxic requests using a pre-trained text classifier before they reach the LLM'}

## Use the step in a serving function

Add `ToxicityGuardrailStep` as the first step in an async serving graph.
Any request whose toxicity score meets or exceeds the threshold will be rejected
before reaching downstream steps.

In [ ]:
project = mlrun.get_or_create_project("toxicity-guardrail-demo", "./toxicity-guardrail-demo")

> 2026-04-27 12:00:00,000 [info] Project loaded successfully: {"project_name":"toxicity-guardrail-demo"}


In [ ]:
fn = project.set_function(
    hub_step.get_src_file_path(),
    name="guardrail-fn",
    kind="serving",
    image="mlrun/mlrun",
    requirements=["transformers", "torch"],
)
graph = fn.set_topology("flow", engine="async")
graph.to(
    class_name="ToxicityGuardrailStep",
    name="toxicity_guardrail",
    threshold=0.5,
).respond()

In [ ]:
project.deploy_function(fn)

### Test with a safe input

In [ ]:
serving_fn = project.get_function("guardrail-fn")
event = {"question": "What is the capital of France?"}
result = serving_fn.invoke("/", body=event)
print("Response:", result)

Response: {'question': 'What is the capital of France?'}


### Test with a toxic input (expect a block)

In [ ]:
try:
    result = serving_fn.invoke("/", body={"question": "some toxic text"})
    print("Response:", result)
except Exception as e:
    print(f"Blocked (expected): {e}")

Blocked (expected): bad function response 500: ValueError: Request blocked: toxicity score 0.998 >= 0.5


## Add the step directly from the hub

If no customisation is needed, the step can be referenced directly from the hub
without downloading the source file first.

In [ ]:
fn2 = project.set_function(
    name="guardrail-fn-2",
    kind="serving",
    image="mlrun/mlrun",
    requirements=["transformers", "torch"],
)
graph2 = fn2.set_topology("flow", engine="async")
graph2.add_step(
    class_name="hub://toxicity_guardrail",
    name="toxicity_guardrail",
    threshold=0.5,
).respond()